# Robustness Analysis 

## Restatement of Main Result

Our primary econometric analysis investigates the relationship between baseline graduate salary in 2018 and changes in full-time employment (FTE) between 2018 and 2020 across Australian university study areas. **We find a negative conditional correlation between higher baseline salary and changes in FTE during the COVID-19 period.**

This is a **descriptive** analysis, not a causal claim.

## Robustness Plan

To test the stability of our main finding, we implement several robustness checks:
- We estimate alternative functional forms (using log of salary).
- We restrict the sample (exclude outliers and health-related study areas).
- We summarize all checks side-by-side in a robustness table with clear interpretation.

In [8]:
import pandas as pd 
import numpy as np 
import statsmodels.api as sm 
import statsmodels.formula.api as smf 

In [9]:
df = pd.read_csv("../data/clean/final_pandemic_research_data.csv")

df.head()

,Study_Area,Salary_18,Salary_20,FTE_18,FTE_20,Salary_Diff,FTE_Diff
0,Science and mathematics,61000,64000.0,64.6,59.1,3000.0,-5.5
1,Computing and Information Systems,60000,65000.0,73.2,72.1,5000.0,-1.1
2,Engineering,65000,69500.0,83.1,83.0,4500.0,-0.1
3,Architecture and built environment,58700,64700.0,77.7,67.7,6000.0,-10.0
4,Agriculture and environmental studies,58300,61500.0,68.3,67.4,3200.0,-0.9


In [11]:
df = df[
    ~df["Study_Area"].isin(["All", "Standard deviation"])
]

print(df["Study_Area"])
print(df.shape)

0                               Science and mathematics
1                     Computing and Information Systems
2                                           Engineering
3                    Architecture and built environment
4                 Agriculture and environmental studies
5                           Health services and support
6                                              Medicine
7                                               Nursing
8                                              Pharmacy
9                                             Dentistry
10                                   Veterinary science
11                                       Rehabilitation
12                                    Teacher education
13                              Business and management
14              Humanities, culture and social sciences
15                                          Social work
16                                           Psychology
17                            Law and paralegal 

In [13]:
model1 = smf.ols(
    "FTE_Diff ~ Salary_18",
    data=df
).fit(cov_type="HC3")

print(model1.summary())

                            OLS Regression Results                            
Dep. Variable:               FTE_Diff   R-squared:                       0.016
Model:                            OLS   Adj. R-squared:                 -0.036
Method:                 Least Squares   F-statistic:                    0.2795
Date:                Fri, 08 May 2026   Prob (F-statistic):              0.603
Time:                        14:48:29   Log-Likelihood:                -51.040
No. Observations:                  21   AIC:                             106.1
Df Residuals:                      19   BIC:                             108.2
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.7393      5.379     -0.323      0.7

In [14]:
df["log_salary"] = np.log(df["Salary_18"])

In [15]:
model2 = smf.ols(
    "FTE_Diff ~ log_salary",
    data=df
).fit(cov_type="HC3")

print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:               FTE_Diff   R-squared:                       0.012
Model:                            OLS   Adj. R-squared:                 -0.040
Method:                 Least Squares   F-statistic:                    0.1810
Date:                Fri, 08 May 2026   Prob (F-statistic):              0.675
Time:                        14:48:33   Log-Likelihood:                -51.086
No. Observations:                  21   AIC:                             106.2
Df Residuals:                      19   BIC:                             108.3
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     22.6545     63.964      0.354      0.7

## Robustness Check 1: Alternative Functional Form - Logarithmic Salary Specification 

This robustness check replaces baseline salary with the logarithm of salary to reduce the influence of extremely high-income study areas and possible skewness in salary values. 

After removing the aggregate rows ("All" and "Standard deviation"), the coefficient on log salary remained negative but became statistically insignificant. This suggests that the earlier relationship was partly influenced by the inclusion of non-study-area observations.

The result indicates that the negative relationship between baseline salary and changes in full-time employment is not very stable across alternative specifications. Therefore, the main descriptive finding should be interpreted cautiously.  

In [16]:
q1 = df["Salary_18"].quantile(0.25)
q3 = df["Salary_18"].quantile(0.75)

iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr 

df_no_outliers = df[
    (df["Salary_18"] >= lower) &
    (df["Salary_18"] <= upper)
]

print(df_no_outliers[["Study_Area", "Salary_18"]]) 
print(df_no_outliers.shape)

                                           Study_Area  Salary_18
0                             Science and mathematics      61000
1                   Computing and Information Systems      60000
2                                         Engineering      65000
3                  Architecture and built environment      58700
4               Agriculture and environmental studies      58300
5                         Health services and support      62600
7                                             Nursing      61600
10                                 Veterinary science      55000
11                                     Rehabilitation      62600
12                                  Teacher education      65500
13                            Business and management      58000
14            Humanities, culture and social sciences      58400
15                                        Social work      65600
16                                         Psychology      60000
17                       

In [17]:
model3 = smf.ols(
    "FTE_Diff ~ Salary_18",
    data=df_no_outliers
).fit(cov_type="HC3")

print(model3.summary())

                            OLS Regression Results                            
Dep. Variable:               FTE_Diff   R-squared:                       0.194
Model:                            OLS   Adj. R-squared:                  0.140
Method:                 Least Squares   F-statistic:                     4.310
Date:                Fri, 08 May 2026   Prob (F-statistic):             0.0555
Time:                        15:04:08   Log-Likelihood:                -39.203
No. Observations:                  17   AIC:                             82.41
Df Residuals:                      15   BIC:                             84.07
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    -23.4514      9.092     -2.579      0.0

## Robustness Check 2: Alternative Sample - Excluding Salary Outliers

This robustness check removes study areas with unusually high or low salary values using the interquartile range (IQR) method. The purpose is to test whether the main result is driven by a small number of extreme observations. 

After excluding outliers, the coefficient on baseline salary became positive and statistically significant. This differs from the earlier specifications, where the relationship was negative and statistically insignificant. 

The result suggests that the direction and strength of the relationship are sensitive to the inclusion of extreme study areas such as Medicine and Dentistry. Therefore, the main descriptive finding does not appear fully stable across alternative samples and should be interpreted cautiously.

In [18]:
df_non_health = df[
    ~df["Study_Area"].str.contains(
        "Health|Medicine|Nursing|Pharmacy|Dentistry|Rehabilitation",
        case=False,
        na=False
    )
]

print(df_non_health["Study_Area"])
print(df_non_health.shape)

0                               Science and mathematics
1                     Computing and Information Systems
2                                           Engineering
3                    Architecture and built environment
4                 Agriculture and environmental studies
10                                   Veterinary science
12                                    Teacher education
13                              Business and management
14              Humanities, culture and social sciences
15                                          Social work
16                                           Psychology
17                            Law and paralegal studies
18                                        Creative arts
19                                       Communications
20    Tourism, Hospitality, Personal Services, Sport...
Name: Study_Area, dtype: str
(15, 8)


In [19]:
model4 = smf.ols(
    "FTE_Diff ~ Salary_18",
    data=df_non_health
).fit(cov_type="HC3")

print(model4.summary())

                            OLS Regression Results                            
Dep. Variable:               FTE_Diff   R-squared:                       0.245
Model:                            OLS   Adj. R-squared:                  0.187
Method:                 Least Squares   F-statistic:                     4.716
Date:                Fri, 08 May 2026   Prob (F-statistic):             0.0490
Time:                        15:13:48   Log-Likelihood:                -34.749
No. Observations:                  15   AIC:                             73.50
Df Residuals:                      13   BIC:                             74.91
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    -22.8128      8.189     -2.786      0.0

## Robustness Check 3: Excluding Health - Related Disciplines 

This robustness check excludes health-related study areas because the COVID-19 pandemic may have affected healthcare sectors differently from other industries. 

After removing health-related disciplines, the coefficient on baseline salary became positive and statistically significant. This suggests that among non-health study areas, higher baseline salaries were associated with better changes in full-time employment outcomes during the pandemic period. 

The result differs from the baseline specification, indicating that the overall relationship is sensitive to the inclusion of health-related disciplines. Therefore, the maindescriptive finding should be interpreted carefully, as different subsamples produce different conclusions. 

In [23]:
import sys
!{sys.executable} -m pip install stargazer


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip3 install --upgrade pip


In [25]:
from stargazer.stargazer import Stargazer
from IPython.core.display import HTML

In [26]:
from stargazer.stargazer import Stargazer

robustness_table = Stargazer([model1, model2, model3, model4])

robustness_table.title("Robustness Analysis")
robustness_table.custom_columns(
    ["Baseline", "Log Salary", "No Outliers", "Non-Health Sample"],
    [1, 1, 1, 1]
)

robustness_table.show_model_numbers(False)

robustness_table.covariate_order([
    "Salary_18",
    "log_salary"
])

robustness_table.rename_covariates({
    "Salary_18": "Baseline Salary (2018)",
    "log_salary": "Log Salary"
})

HTML(robustness_table.render_html())

### Notes

- Column 1 reports the baseline specification using the full cleaned sample of study areas.
- Column 2 replaces baseline salary with the logarithm of salary to test an alternative functional form.
- Column 3 excludes salary outliers using the interquartile range (IQR) method.
- Column 4 excludes health-related study areas to test whether the relationship is driven by pandemic-sensitive sectors.
- All regressions use HC3 heteroskedasticity-robust standard errors; Observations (N) is number of study areas in each specification. 

# Overall Robustness Interpretation

The robustness checks show that the relationship between baseline salary and changes in full-time employment is sensitive to specification choice and sample composition. 

The baseline and logarithmic salary models produced negative but statistically insiginificant relationships. However, after removing salary outliers or excluding health-related disciplines, the coefficient became positive and statistically significant. 

These results suggest that the overall relationship is influenced by a small number of extreme study areas and by sectors that were uniquely affected during the COVID-19 period. Therefore, the main descriptive finding should be interpreted cautiously, as different specifications produce different conclusions. 